**Ingest Circuits - Incremental**

Reads `circuits.csv` from the batch landing folder, adds metadata, and writes to `formula1_incr.bronze.circuits` partitioned by `batch_id`.

**Read CSV with explicit schema**

In [0]:
%run ../00-common/01.environment-config

In [0]:
dbutils.widgets.text('p_batch_id','')
v_batch_id = dbutils.widgets.get('p_batch_id')

In [0]:
import pyspark.sql.functions as f

In [0]:
Source_table = f'{loding_folder_path}/{v_batch_id}/circuits.csv'
table_name = f'{catalog_name}.{bronze_schema}.circuits'

In [0]:
from pyspark.sql.types import *
circutes_schema = StructType([
  StructField('circuitId', StringType(), True),
  StructField('url', StringType(), True),
  StructField('circuitName', StringType(), True),
  StructField('lat', DoubleType(),True),
  StructField('long', DoubleType(), True),
 StructField('locality', StringType(), True),
  StructField('country', StringType(), True)
])

In [0]:
circuits_df = (
  spark.read
  .format('csv')
  .option('header', True)
  #.option('mode', 'FAILFAST')
  # .option ('inferSchema','True')
  .schema(circutes_schema)
  .load(Source_table))

In [0]:
display(circuits_df)

**Schema definition**

In [0]:
display(circutes_schema)

**Add metadata columns** (`ingestion_timestamp`, `source_file`)

In [0]:
from pyspark.sql.functions import *
circuits_final_df = (
 (circuits_df
 .withColumn('ingestion_timestamp', current_timestamp())
 .withColumn('source_file', col('_metadata.file_path'))))
display(circuits_final_df)

**Write to bronze Delta table** (overwrite per batch partition)

In [0]:
circuits_final_df = circuits_final_df.withColumn('batch_id', f.lit(v_batch_id))

In [0]:
(
circuits_final_df
    .write
    .format('delta')
    .mode('overwrite')
    .partitionBy('batch_id')
    .option('replaceWhere', f'batch_id = "{v_batch_id}"')
    .saveAsTable(table_name) 
    # table name is catalog, schema and table name so u can replace it with your own predefined variables see the next page for race file 
)

In [0]:
display(circuits_final_df)